# 01 USAspending pilot: PIDP, BRIC and EDA in coastal CA/OR/WA

**Task:** T1.1 (Coastal Harbor Funding Equity)

**Reads:** `data/processed/usaspending_pilot/{prime_awards,prime_transactions,subawards}.csv` and `process_log.json`, written by `uv run python -m harborfund.ingest.usaspending all`. Also reads `data/reference/coastal_counties.csv`.

**Writes:** nothing. The outputs are the pilot report, so they are kept when committed. They hold aggregates only, no raw records.

**Scope used here:**
- PIDP and EDA: an award counts if its recipient county or place-of-performance county is one of the 42 NOAA coastal shoreline counties.
- BRIC (97.047) prime awards: every prime in CA/OR/WA counts, since the recipient is usually the state. BRIC subawards are county-flagged from ZIP codes.
- Window FY2010–FY2026 (BRIC FY2020+). Award-level tables use the base action FY. Dollar tables by FY use transaction obligations by action FY.
- Dollars are nominal.

In [1]:
import json
import numpy as np
import pandas as pd
from harborfund import config

OUT = config.PROCESSED_DIR / "usaspending_pilot"
log = json.loads((OUT / "process_log.json").read_text())
BOOL = ["in_fy_window", "has_uei", "has_duns", "recipient_is_state_gov", "prime_recipient_is_state_gov",
        "state_level_prime", "recipient_in_coastal_county", "pop_in_coastal_county", "in_coastal_county",
        "in_states", "earmark_text_flag", "earmark_candidate_year", "earmark_flag", "eda_rlf_flag",
        "harbor_kw_flag", "in_scope", "zip_multi_county", "zip_unmatched"]

def load(name):
    df = pd.read_csv(OUT / f"{name}.csv", dtype=str, low_memory=False)
    for c in BOOL:
        if c in df:
            df[c] = df[c].map({"True": True, "False": False}).fillna(False).astype(bool)
    df["fy"] = pd.to_numeric(df["fy"], errors="coerce").astype("Int64")
    for c in [c for c in df if c.endswith("_nominal")]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ("recipient_state", "pop_state", "geo_match_basis"):
        df[c] = df[c].fillna("")
    # State the row is counted in: the matched coastal county's state; otherwise
    # (state-level BRIC primes) the recipient state if in CA/OR/WA, else the POP state.
    rec_ok = df["recipient_state"].isin(["CA", "OR", "WA"])
    df["geo_state"] = np.select(
        [df["recipient_in_coastal_county"], df["pop_in_coastal_county"], rec_ok],
        [df["recipient_state"], df["pop_state"], df["recipient_state"]], default=df["pop_state"])
    return df

awards, txns, subs = (load(n) for n in ("prime_awards", "prime_transactions", "subawards"))
print("raw run:", log["raw_run"], "| processed:", log["processed_utc"])
pd.DataFrame({k: v["raw_rows"] | {"kept_" + kk: vv for kk, vv in v["in_scope_rows"].items()}
              for k, v in log["downloads"].items()}).T

raw run: data/raw/usaspending/2026-11-18 | processed: 2026-11-18T14:24:18+00:00


,prime_awards,prime_transactions,subawards,kept_prime_awards,kept_prime_transactions,kept_subawards
aln_20.823,146,226,7,33,47,4
aln_11.300,1145,1156,36,33,33,0
aln_11.307,4284,5171,1081,193,234,32
aln_97.047,1275,2742,1360,132,300,73


## 1. Scope and award types

In [2]:
A = awards[awards.in_scope & awards.in_fy_window].copy()
T = txns[txns.in_scope & txns.in_fy_window].copy()
S = subs[subs.in_fy_window].copy()
print(f"prime awards in scope: {len(A):,} | transactions: {len(T):,} | subawards in CA/OR/WA: {len(S):,} "
      f"(in coastal counties: {S.in_coastal_county.sum():,})")
pd.crosstab(A["program_id"], A["assistance_type_description"].fillna("(blank)"), margins=True)

prime awards in scope: 351 | transactions: 614 | subawards in CA/OR/WA: 228 (in coastal counties: 109)


assistance_type_description,(blank),FORMULA GRANT (A),PROJECT GRANT (B),All
program_id,,,,
bric,0,8,62,70
eda_eaa,50,0,142,192
eda_public_works,21,0,11,32
pdm_cpf,0,0,2,2
pdm_other,0,0,22,22
pidp,0,0,33,33
All,71,8,272,351


## 2. Row counts and nominal totals by program × state × FY
Awards are counted by base action FY, and the dollars are each award's total obligation to date. The second table sums transaction obligations by action FY.

In [3]:
t_awards = (A.groupby(["program_id", "geo_state", "fy"])
              .agg(awards=("assistance_award_unique_key", "nunique"), total_obligated_nominal=("total_obligated_nominal", "sum"))
              .reset_index())
pd.set_option("display.max_rows", 400)
t_awards.pivot_table(index=["program_id", "geo_state"], columns="fy", values="awards", fill_value=0, aggfunc="sum")

fy                          2010  2011  2012  2013  2014  2015  2016  2017  \
program_id       geo_state                                                   
bric             CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
eda_eaa          CA            5     8     6     2     4     2     3     3   
                 OR            1     2     1     0     0     0     0     0   
                 WA            2     5     0     1     1     1     1     1   
eda_public_works CA            2     0     3     0     0     3     3     0   
                 OR            1     0     0     0     0     0     0     2   
                 WA            3     1     1     0     1     1     0     0   
pdm_cpf          WA            0     0     0     0     0     0     0     0   
pdm_other        CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   
pidp             CA            0     0     0     0     0     0     0     0   
                 OR            0     0     0     0     0     0     0     0   
                 WA            0     0     0     0     0     0     0     0   

fy                          2018  2019  2020  2021  2022  2023  2024  2025  \
program_id       geo_state                                                   
bric             CA            0     0     0     0     4     7    15     2   
                 OR            0     0     0     0     2     0     2     0   
                 WA            0     0     0     0    16    16     3     1   
eda_eaa          CA            5    10    23    24    16     1    10     2   
                 OR            0     0     5     2     7     0     0     0   
                 WA            0     0     9     8    13     0     3     4   
eda_public_works CA            2     0     0     2     1     1     0     1   
                 OR            0     0     1     1     1     0     0     0   
                 WA            0     0     0     0     1     0     0     0   
pdm_cpf          WA            0     0     0     0     2     0     0     0   
pdm_other        CA            0     0     8     4     0     0     0     0   
                 OR            0     0     1     1     0     0     0     0   
                 WA            0     0     4     4     0     0     0     0   
pidp             CA            0     0     0     0     2     2     6     2   
                 OR            0     0     0     0     0     1     1     1   
                 WA            0     0     1     0     1     3     2     3   

fy                          2026  
program_id       geo_state        
bric             CA            0  
                 OR            0  
                 WA            2  
eda_eaa          CA            0  
                 OR            0  
                 WA            1  
eda_public_works CA            0  
                 OR            0  
                 WA            0  
pdm_cpf          WA            0  
pdm_other        CA            0  
                 OR            0  
                 WA            0  
pidp             CA            5  
                 OR            0  
                 WA            3

In [4]:
(T.groupby(["program_id", "geo_state", "fy"])["federal_action_obligation_nominal"].sum()
   .unstack("fy").fillna(0).div(1e6).round(2))  # $M nominal, by action FY

fy                          2010  2011  2012  2013  2014  2015  2016  2017  \
program_id       geo_state                                                   
bric             CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
eda_eaa          CA         4.61  8.06  5.25   1.0  2.63  2.65  1.73  1.97   
                 OR         2.75  1.32  1.76   0.0  0.50  0.00  0.00  0.00   
                 WA         0.21  2.18  0.00   2.0  0.15  0.20  0.04  0.20   
eda_public_works CA         6.30  0.00  6.75   0.0  0.00  7.10  7.74  0.00   
                 OR         0.76  0.00  0.00   0.0  0.00  0.00  0.00  4.92   
                 WA         6.20  2.60  1.00   0.0  1.50  2.00  0.00  0.00   
pdm_cpf          WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
pdm_other        CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
pidp             CA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 OR         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   
                 WA         0.00  0.00  0.00   0.0  0.00  0.00  0.00  0.00   

fy                          2018   2019   2020   2021   2022   2023    2024  \
program_id       geo_state                                                    
bric             CA         0.00   0.00   0.00   0.00   2.17  26.18   47.96   
                 OR         0.00   0.00   0.00   0.00   4.40  -0.10    5.08   
                 WA         0.00   0.00   0.00   0.00   5.61  35.22   27.79   
eda_eaa          CA         4.07  24.79  65.43  32.53  35.44  -4.51    6.63   
                 OR         0.00   0.00   3.50   2.24  28.87   0.00    0.00   
                 WA         0.00   0.00  22.71   9.59  87.93   0.00    3.90   
eda_public_works CA         6.00   0.00   0.00   3.86   2.66   6.02    0.00   
                 OR         0.00   0.00   1.20   2.75   0.56   0.00    0.00   
                 WA         0.00   0.00   0.00   0.00   2.25   0.00    0.00   
pdm_cpf          WA         0.00   0.00   0.00   0.00  12.58   0.00    0.00   
pdm_other        CA         0.00   0.00  13.94   4.54   0.00   0.00    0.00   
                 OR         0.00   0.00   0.42   4.25   0.00   0.00    0.00   
                 WA         0.00   0.00  14.91   3.43   0.00   0.00    0.00   
pidp             CA         0.00   0.00   0.00   0.00  32.68   9.88  106.57   
                 OR         0.00   0.00   0.00   0.00   0.00   9.88    3.44   
                 WA         0.00   0.00   0.00   0.00  10.69  41.37   27.94   

fy                           2025    2026  
program_id       geo_state                 
bric             CA          4.83   -1.60  
                 OR          4.32    0.24  
                 WA         15.82   34.63  
eda_eaa          CA          2.85    0.00  
                 OR          0.00    0.00  
                 WA          9.26    6.20  
eda_public_works CA          4.24    0.00  
                 OR          0.00    0.00  
                 WA          0.00    0.00  
pdm_cpf          WA          0.00    0.00  
pdm_other        CA          0.00    0.00  
                 OR          0.00    0.00  
                 WA          0.00    0.00  
pidp             CA          7.55  114.72  
                 OR          7.73   -0.01  
                 WA         70.36   20.05

In [5]:
summary = (A.groupby(["program_id", "geo_state"])
             .agg(awards=("assistance_award_unique_key", "nunique"),
                  obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6))
             .round(2))
summary

awards  obligated_m
program_id       geo_state                     
bric             CA             28        81.63
                 OR              4        10.22
                 WA             38       138.30
eda_eaa          CA            124       194.75
                 OR             18        40.21
                 WA             50       144.07
eda_public_works CA             18        50.39
                 OR              6        10.18
                 WA              8        15.55
pdm_cpf          WA              2        -0.04
pdm_other        CA             12         6.65
                 OR              2         8.79
                 WA              8        19.09
pidp             CA             17       271.40
                 OR              3        21.04
                 WA             13       171.92

## 3. Recipient identifiers (UEI / legacy DUNS)
SAM.gov switched from DUNS to UEI in April 2022, so older awards may have only a DUNS.

In [6]:
ids = A.assign(uei_or_duns=A.has_uei | A.has_duns)
pd.concat([
    ids.groupby("program_id")[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3),
    ids.groupby("program_id").size().rename("awards"),
], axis=1)

,has_uei,has_duns,uei_or_duns,awards
program_id,,,,
bric,1.0,0.057,1.0,70
eda_eaa,1.0,0.646,1.0,192
eda_public_works,1.0,0.875,1.0,32
pdm_cpf,1.0,1.000,1.0,2
pdm_other,1.0,0.455,1.0,22
pidp,1.0,0.030,1.0,33


In [7]:
ids.assign(era=pd.cut(ids.fy.astype(float), [2009, 2021, 2026], labels=["FY2010-21", "FY2022-26"])) \
   .groupby(["program_id", "era"], observed=True)[["has_uei", "has_duns", "uei_or_duns"]].mean().round(3)

has_uei  has_duns  uei_or_duns
program_id       era                                      
bric             FY2022-26      1.0     0.057          1.0
eda_eaa          FY2010-21      1.0     0.881          1.0
                 FY2022-26      1.0     0.088          1.0
eda_public_works FY2010-21      1.0     1.000          1.0
                 FY2022-26      1.0     0.200          1.0
pdm_cpf          FY2022-26      1.0     1.000          1.0
pdm_other        FY2010-21      1.0     0.455          1.0
pidp             FY2010-21      1.0     0.000          1.0
                 FY2022-26      1.0     0.031          1.0

In [8]:
sub_ids = S[S.in_coastal_county]
sub_ids.groupby("program_id")[["has_uei", "has_duns"]].mean().round(3).assign(subawards=sub_ids.groupby("program_id").size())

,has_uei,has_duns,subawards
program_id,,,
bric,1.0,0.981,53
eda_eaa,1.0,0.781,32
pdm_cpf,1.0,1.000,4
pdm_other,1.0,1.000,16
pidp,1.0,1.000,4


## 4. BRIC: state-held prime dollars and what subawards identify

In [9]:
B = A[A.aln == "97.047"]
bric_state = B.groupby("recipient_is_state_gov")["total_obligated_nominal"].agg(["count", "sum"])
bric_state["share_of_dollars"] = (bric_state["sum"] / bric_state["sum"].sum()).round(3)
bric_state

,count,sum,share_of_dollars
recipient_is_state_gov,,,
False,62,45781734.1,0.173
True,32,218874103.1,0.827


In [10]:
state_primes = B[B.recipient_is_state_gov]
sb = subs[(subs.aln == "97.047") & subs.prime_award_unique_key.isin(state_primes.assistance_award_unique_key)]
rows = []
for st, g in state_primes.groupby("recipient_state"):
    s_all = sb[sb.prime_award_unique_key.isin(g.assistance_award_unique_key)]
    s_coast = s_all[s_all.in_coastal_county]
    prime_d = g.total_obligated_nominal.sum()
    rows.append({"state": st, "state_primes": len(g), "prime_obligated_m": prime_d / 1e6,
                 "subawards": len(s_all), "subaward_m": s_all.subaward_amount_nominal.sum() / 1e6,
                 "share_identified_by_subawards": s_all.subaward_amount_nominal.sum() / prime_d if prime_d else None,
                 "coastal_subawards": len(s_coast), "coastal_subaward_m": s_coast.subaward_amount_nominal.sum() / 1e6})
pd.DataFrame(rows).round(3)

,state,state_primes,prime_obligated_m,subawards,subaward_m,share_identified_by_subawards,coastal_subawards,coastal_subaward_m
0,CA,11,63.874,50,57.610,0.902,33,41.902
1,OR,6,19.017,16,11.037,0.580,4,4.334
2,WA,15,135.983,42,118.457,0.871,24,100.542


Caveat: subaward amounts are FFATA-reported and can be double-reported or amended, so the identified share may go above 1. The prime total is the obligation to date.

In [11]:
sub_geo = S[S.aln == "97.047"]
pd.Series({"subawards_in_states": len(sub_geo), "zip_unmatched": sub_geo.zip_unmatched.sum(),
           "zip_multi_county": sub_geo.zip_multi_county.sum(), "in_coastal_county": sub_geo.in_coastal_county.sum()})

subawards_in_states    133
zip_unmatched           10
zip_multi_county        13
in_coastal_county       73
dtype: int64

## 5. Earmark flags
These are text-based flags for the pilot. The definitive earmark tags come from the appropriations CPF/CDS tables (T2.4). The funding-opportunity table below shows which FONs carry the earmarks, so the rule can be tightened after the first run.

In [12]:
E = A[A.aln.isin(["20.823", "97.047"])]
(E.groupby(["aln", "program_id", "program_basis"])
   .agg(awards=("assistance_award_unique_key", "nunique"), obligated_m=("total_obligated_nominal", lambda s: s.sum() / 1e6),
        earmark_flagged=("earmark_flag", "sum"))
   .round(2))

awards  obligated_m  earmark_flagged
aln    program_id program_basis                                      
20.823 pidp       default            33       464.36                0
97.047 bric       default             1        19.90                0
                  text:BRIC          69       210.26                0
       pdm_cpf    text:CPF/PDM        2        -0.04                2
       pdm_other  text:PDM           22        34.54                0

In [13]:
E.assign(fon=E.funding_opportunity_number.fillna("(blank)")) \
 .pivot_table(index=["aln", "fon"], columns="fy", values="assistance_award_unique_key", aggfunc="nunique", fill_value=0)

fy                          2020  2021  2022  2023  2024  2025  2026
aln    fon                                                          
20.823 (blank)                 0     0     1     2     0     0     0
       693JF7-19-BAA-0002      0     0     1     0     0     0     0
       693JF7-20-BAA-0001      1     0     0     1     0     0     0
       693JF719BAA0002         0     0     1     0     0     0     0
       693JF720BAA0001         0     0     0     1     0     0     0
       693JF722R000012         0     0     0     0     0     1     0
       MA PID 22 001           0     0     0     0     1     0     0
       MA PID-22 001           0     0     0     0     1     0     0
       MA-PID-21-001           0     0     0     1     1     0     0
       MA-PID-22-001           0     0     0     1     2     1     3
       MA-PID-23-001           0     0     0     0     0     2     0
       MA-PID-24-001           0     0     0     0     0     1     4
       MAPID21001              0     0     0     0     1     0     0
       MAPID22001              0     0     0     0     1     0     0
       MAPID23001              0     0     0     0     2     1     1
97.047 (blank)                 7     4     9    13     0     0     0
       -                       5     4     1     0     2     1     0
       DHS-20-MT-047-00-99     0     0    13    10     8     0     1
       DHS-21-MT-047-00-99     0     0     0     0     2     0     0
       DHS-22-MT-047-00-98     0     0     0     0     6     0     1
       DHS-22-MT-047-00-99     0     0     0     0     2     0     0
       DHS-23-MT-047-00-98     0     0     0     0     0     2     0
       NOT APPLICABLE          1     1     1     0     0     0     0

In [14]:
E[E.earmark_flag][["aln", "program_id", "fy", "recipient_name", "geo_state", "total_obligated_nominal",
                   "funding_opportunity_number", "prime_award_base_transaction_description"]].sort_values(["aln", "fy"])

,aln,program_id,fy,recipient_name,geo_state,total_obligated_nominal,funding_opportunity_number,prime_award_base_transaction_description
621,97.047,pdm_cpf,2022,"MILITARY DEPARTMENT, WASHINGTON STATE",WA,-1.0,NaN,PRE-DISASTER MITIGATION
631,97.047,pdm_cpf,2022,SPOKANE TRIBE OF THE SPOKANE RESERVATION,WA,-42030.0,NaN,PRE-DISASTER MITIGATION


## 6. Harbor keyword flag and geography match basis
`harbor_kw_flag` applies only the inventory's keyword rule. The crosswalk part of the rule comes later (T1.2/T1.3). PIDP is `all` (a port program).

In [15]:
pd.concat([A.groupby("program_id")["harbor_kw_flag"].agg(["sum", "mean"]).rename(columns={"sum": "flagged", "mean": "share"}),
           A[A.harbor_kw_flag].groupby("program_id")["total_obligated_nominal"].sum().div(1e6).rename("flagged_m")], axis=1).round(3)

,flagged,share,flagged_m
program_id,,,
bric,0,0.000,NaN
eda_eaa,53,0.276,177.787
eda_public_works,12,0.375,29.023
pdm_cpf,0,0.000,NaN
pdm_other,0,0.000,NaN
pidp,33,1.000,464.364


In [16]:
pd.crosstab(A.program_id, A.geo_match_basis.replace("", "(state-level BRIC prime)"), margins=True)

geo_match_basis,(state-level BRIC prime),both,pop,recipient,All
program_id,,,,,
bric,25,45,0,0,70
eda_eaa,0,172,5,15,192
eda_public_works,0,32,0,0,32
pdm_cpf,1,1,0,0,2
pdm_other,8,14,0,0,22
pidp,0,33,0,0,33
All,34,297,5,15,351


In [17]:
if "eda_rlf_flag" in A:
    print(A[A.aln.isin(["11.300", "11.307"])].groupby("program_id")["eda_rlf_flag"].sum())

program_id
eda_eaa             30
eda_public_works     0
Name: eda_rlf_flag, dtype: int64


## 7. Independent count checks
One program-year per program, compared with a published figure. Each external number and its source are filled in from a document I read, and the `source` string says which one. A difference is explained in the handoff, not forced to zero.

In [18]:
EXTERNAL_CHECKS = [
    # {"program_id": ..., "fy": ..., "states": [...], "external_count": ..., "source": "..."}
]

def our_count(pid, fy, states):
    g = awards[(awards.program_id.str.startswith(pid)) & (awards.fy == fy) & awards.in_states &
               (awards.recipient_state.isin(states) | awards.pop_state.isin(states))]
    return g.assistance_award_unique_key.nunique(), g.total_obligated_nominal.sum()

pd.DataFrame([{**c, "usaspending_count": our_count(c["program_id"], c["fy"], c["states"])[0],
               "usaspending_obligated": our_count(c["program_id"], c["fy"], c["states"])[1]}
              for c in EXTERNAL_CHECKS])

""
